# NeuroSeg-X: Brain MRI Multi-Task Framework
### VERSION: 2.2 (STABLE ASCII + ROBUST DATA SCANNER)

**Fixes applied:**
1. **Unicode Fix**: Absolutely zero non-ASCII characters to prevent Colab crashes.
2. **Robust Scanner**: Now handles cases where masks are in separate subfolders (e.g. /images and /masks).
3. **File Support**: Supports PNG, JPG, JPEG, and NIfTI detection.

In [ ]:
# -------------------------------------------------------------------
# STEP 1: INSTALLS & IMPORTS
# -------------------------------------------------------------------
!pip install -q albumentations opencv-python-headless tensorboard tqdm torchmetrics scikit-learn

import os
import random
import shutil
import zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm 
from PIL import Image
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from google.colab import drive, files

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, jaccard_score

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# -------------------------------------------------------------------
# STEP 2: CONFIGURATION
# -------------------------------------------------------------------
config = {
    'image_size': [512, 512],
    'batch_size': 4,
    'num_epochs': 50,
    'lr': 1e-4,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'save_dir': '/content/results',
    'data_dir': '/content/data',
    'seed': 42
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(config['seed'])
os.makedirs(config['save_dir'], exist_ok=True)
print("[OK] Configuration saved.")

In [ ]:
# -------------------------------------------------------------------
# STEP 3: DATA EXTRACTION
# -------------------------------------------------------------------
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/Colab_Notebooks_Data'
if not os.path.exists(drive_path): drive_path = '/content/drive/MyDrive/Colab_Notebook_Data'

datasets = ['brain_glioma.zip', 'brain_menin.zip', 'brain_tumor.zip']
raw_extract = '/content/raw_data'
os.makedirs(raw_extract, exist_ok=True)

print(f"[INFO] Looking for ZIP files in: {drive_path}")
for ds in datasets:
    zp = os.path.join(drive_path, ds)
    if os.path.exists(zp):
        print(f"  - Extracting {ds}...")
        with zipfile.ZipFile(zp, 'r') as z: z.extractall(raw_extract)
    else:
        print(f"  [WARN] {ds} not found!")

print("[OK] Extraction complete.")

In [ ]:
# -------------------------------------------------------------------
# STEP 4: ROBUST DATA ORGANIZATION (SUPER SCANNER)
# -------------------------------------------------------------------
print("[INFO] Starting Robust Data Discovery...")

data_root = Path(config['data_dir'])
for split in ['train', 'val']: 
    (data_root / split / 'images').mkdir(parents=True, exist_ok=True)
    (data_root / split / 'masks').mkdir(parents=True, exist_ok=True)

all_samples = []
raw_p = Path('/content/raw_data')

# Find all potential image files
all_files = list(raw_p.rglob('*'))
image_files = [f for f in all_files if f.suffix.lower() in ['.png', '.jpg', '.jpeg', '.tif', '.nii']]

print(f"[DEBUG] Found {len(image_files)} total image-like files.")

for img_p in tqdm(image_files, desc="Pairing images/masks", ascii=True):
    name_low = img_p.name.lower()
    # Skip files that are clearly masks or system files
    if any(x in name_low for x in ['mask', 'seg', 'label', 'gt']): continue
    if img_p.name.startswith('.'): continue
    
    # Search for a corresponding mask
    # We look for a file in the same extraction folder that contains the image name + a mask suffix
    mask_p = None
    stem = img_p.stem
    
    # Pattern 1: Same directory
    cands = [img_p.parent / f"{stem}_mask.png", img_p.parent / f"{stem}_seg.png", img_p.parent / f"{stem}_mask{img_p.suffix}"]
    # Pattern 2: Parallel 'masks' or 'labels' folder
    parent_of_parent = img_p.parent.parent
    for m_fld in ['masks', 'labels', 'segs', 'segmentation', 'GroundTruth']:
        mask_dir = parent_of_parent / m_fld
        if mask_dir.exists():
            cands.append(mask_dir / f"{stem}.png")
            cands.append(mask_dir / f"{stem}_mask.png")
            cands.append(mask_dir / f"{img_p.name}") # Same name but in masks folder
            
    for c in cands:
        if c.exists():
            mask_p = c
            break
            
    if mask_p:
        # Determine labels from path (glioma = HGG, others = LGG)
        path_str = str(img_p).lower()
        label_grading = 1 if 'glioma' in path_str else 0
        all_samples.append({'img': img_p, 'mask': mask_p, 'grading': label_grading})

print(f"\n[RESULTS] Found and paired {len(all_samples)} MRI samples.")

if len(all_samples) == 0:
    print("!!! FATAL ERROR: No images and masks could be paired. Check your folder structure in /content/raw_data !!!")
    # List first 10 files to help debug
    print("Listing first 10 files in raw_data for debugging:")
    for f in all_files[:10]: print(f"  - {f}")
else:
    # Split and Copy
    random.shuffle(all_samples)
    split_idx = int(0.8 * len(all_samples))
    splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}

    for s_name, s_data in splits.items():
        print(f"  Processing {s_name} set ({len(s_data)} samples)...")
        for sample in tqdm(s_data, ascii=True):
            # Encode labels into filename: g[LABEL]_filename
            unique_name = f"g{sample['grading']}_{sample['img'].name}"
            if unique_name.endswith('.nii'): unique_name += ".png" # Force PNG for consistent loading if needed
            
            try:
                shutil.copy2(sample['img'], data_root / s_name / 'images' / unique_name)
                shutil.copy2(sample['mask'], data_root / s_name / 'masks' / unique_name)
            except Exception as e:
                pass # Skip problematic files

    print("[OK] Data organization complete.")

In [ ]:
# -------------------------------------------------------------------
# STEP 5: DATASET & MODELS
# -------------------------------------------------------------------
class NeuroSegDataset(Dataset):
    def __init__(self, split, transforms=None):
        self.img_dir = Path(config['data_dir']) / split / 'images'
        self.mask_dir = Path(config['data_dir']) / split / 'masks'
        self.files = sorted(list(self.img_dir.glob('*')))
        self.transforms = transforms
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img_path = self.files[idx]
        mask_path = self.mask_dir / img_path.name
        
        # Robust loader
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        
        if mask.max() > 1: mask /= 255.0
        grading = int(img_path.name[1])
        
        if self.transforms: 
            aug = self.transforms(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        return {'image': image, 'mask': mask.unsqueeze(0), 'detection': torch.tensor(1.0), 'grading': torch.tensor(grading)}

def get_tf():
    tr = A.Compose([A.Resize(512, 512), A.HorizontalFlip(), A.Normalize(), ToTensorV2()])
    vl = A.Compose([A.Resize(512, 512), A.Normalize(), ToTensorV2()])
    return tr, vl

class DoubleConv(nn.Module):
    def __init__(self, i, o): 
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU())
    def forward(self, x): return self.c(x)

class NeuroSegX(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = DoubleConv(3, 64)
        self.e2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.seg = nn.Conv2d(128, 1, 1)
        self.det = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 1), nn.Sigmoid())
        self.grd = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 2))
    def forward(self, x):
        x1 = self.e1(x)
        x2 = self.e2(x1)
        return {'segmentation': self.seg(x2), 'detection': self.det(x2), 'grading': self.grd(x2)}

print("[OK] Models and Dataset ready.")

In [ ]:
# -------------------------------------------------------------------
# STEP 6: TRAINING
# -------------------------------------------------------------------
if len(os.listdir(config['data_dir'] + '/train/images')) == 0:
    print("!!! ERROR: Train dataset is empty. Cannot start training. !!!")
else:
    tr_tf, vl_tf = get_tf()
    tr_loader = DataLoader(NeuroSegDataset('train', tr_tf), config['batch_size'], shuffle=True)
    vl_loader = DataLoader(NeuroSegDataset('val', vl_tf), config['batch_size'])

    model = NeuroSegX().to(config['device'])
    opt = optim.AdamW(model.parameters(), config['lr'])
    scaler = GradScaler()
    c_seg, c_det, c_grd = nn.BCEWithLogitsLoss(), nn.BCELoss(), nn.CrossEntropyLoss()

    print(f"[INFO] Starting training for {config['num_epochs']} epochs...")
    for epoch in range(config['num_epochs']):
        model.train()
        tl = 0
        for b in tqdm(tr_loader, desc=f"Epoch {epoch+1}", ascii=True):
            imgs, masks = b['image'].to(config['device']), b['mask'].to(config['device'])
            det_gt, grd_gt = b['detection'].to(config['device']), b['grading'].to(config['device'])
            with autocast():
                out = model(imgs)
                mask_out = F.interpolate(out['segmentation'], size=(512,512), mode='bilinear')
                loss = 0.5*c_seg(mask_out, masks) + 0.25*c_det(out['detection'], det_gt.unsqueeze(1)) + 0.25*c_grd(out['grading'], grd_gt)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item()
        print(f"[INFO] Epoch {epoch+1} Complete. Avg Loss: {tl/len(tr_loader):.4f}")
    
    torch.save(model.state_dict(), '/content/neuroseg_final.pth')
    print("[OK] Model saved.")

In [ ]:
# -------------------------------------------------------------------
# STEP 7: EVALUATION & VISUALIZATION
# -------------------------------------------------------------------
if 'vl_loader' in globals():
    print("[INFO] Evaluating...")
    model.eval()
    dices = []
    with torch.no_grad():
        for batch in tqdm(vl_loader, ascii=True):
            imgs, masks = batch['image'].to(config['device']), batch['mask'].to(config['device'])
            out = model(imgs)
            pred_mask = (torch.sigmoid(F.interpolate(out['segmentation'], (512,512))) > 0.5).float()
            for i in range(imgs.size(0)):
                p, g = pred_mask[i].view(-1), masks[i].view(-1)
                inter = (p * g).sum()
                dices.append(((2.*inter)/(p.sum()+g.sum()+1e-8)).item())
    print(f"[RESULTS] Mean Dice Score: {np.mean(dices):.4f}")

    test_batch = next(iter(vl_loader))
    with torch.no_grad():
        out = model(test_batch['image'].to(config['device']))
        pred = (torch.sigmoid(F.interpolate(out['segmentation'], (512,512))) > 0.5).cpu()

    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1); plt.imshow(test_batch['image'][0].permute(1,2,0)); plt.title("Original")
    plt.subplot(1,2,2); plt.imshow(pred[0,0], cmap='jet'); plt.title("Prediction")
    plt.show()
    files.download('/content/neuroseg_final.pth')
else:
    print("!!! Evaluation skipped (No data found). !!!")